# Tests de integración de `Build y export de flujos`

Notebook orientado a verificar el comportamiento end-to-end de OP-08 `build flows`y OP-09 `export flows`,
considerando resultado tabular, resultado operacional y trazabilidad.

In [44]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "synthetic" 

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:
- imports generales,
- imports del módulo,
- helpers de testing reutilizables,
- smoke de imports,
- y configuración básica de display.

Aquí todavía no se prueban escenarios funcionales de `import_trips_from_dataframe()`;
solo se prepara el entorno de trabajo para las secciones siguientes.

### 0.1 Imports generales

Qué prepara: imports base, filesystem visible, JSON y utilidades de copia para los tests.

In [45]:
from pathlib import Path
import copy
import json
import shutil

import pandas as pd

### 0.2 Imports del módulo 

In [46]:
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema
from pylondrina.datasets import TripDataset, FlowDataset
from pylondrina.reports import FlowBuildReport, OperationReport
from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.validation import ValidationOptions, validate_trips
from pylondrina.transforms.flows import build_flows, FlowBuildOptions
from pylondrina.export.flows import export_flows, ExportFlowsOptions, FlowExportResult
from pylondrina.errors import ExportError, ValidationError

### 0.3 Import del generador sintético

Qué prepara: acceso al generador para crear fixtures ricas.

In [47]:
from scripts.synthetic_data.base_generator import generate_synthetic_trip_dataframe

### 0.4 Helpers de testing reutilizables

In [48]:
def show_ok(label: str):
    print(f"OK - {label}")


def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, ensure_ascii=False, default=str)
    except Exception as exc:
        raise AssertionError(f"{label} no es JSON-safe: {exc}") from exc


def clone_tripdataset(trips: TripDataset) -> TripDataset:
    return copy.deepcopy(trips)


def clone_flowdataset(flows: FlowDataset) -> FlowDataset:
    return copy.deepcopy(flows)

### 0.5 Configuración de display

In [49]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

### 0.6 Carpeta visible de integración

Qué prepara: root visible de artefactos de prueba. No usa carpetas temporales ocultas

In [50]:
IT_ROOT = REPO_ROOT / "notebooks" / "testing" / "build_flows" / "tmp_integration"


def reset_it_root() -> Path:
    if IT_ROOT.exists():
        shutil.rmtree(IT_ROOT)
    IT_ROOT.mkdir(parents=True, exist_ok=True)
    return IT_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = IT_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


root = reset_it_root()
print("IT_ROOT =", root.resolve())
show_ok("Sección 0 lista")

IT_ROOT = D:\Memoria\repos\pylondrina\notebooks\testing\build_flows\tmp_integration
OK - Sección 0 lista


## Sección 1. Fixtures ricas reutilizables
### 1.1 Constantes de schema para fixtures ricas

Qué prepara: required/base fields y dominios suficientes para importar fixtures ricas y luego construir/exportar flows.

In [51]:
REQUIRED_FIELDS_ORDER = [
    "movement_id",
    "user_id",
    "origin_longitude",
    "origin_latitude",
    "destination_longitude",
    "destination_latitude",
    "origin_h3_index",
    "destination_h3_index",
    "origin_time_utc",
    "destination_time_utc",
    "trip_id",
    "movement_seq",
]

REQUIRED_FIELD_DTYPES = {
    "movement_id": "string",
    "user_id": "string",
    "origin_longitude": "float",
    "origin_latitude": "float",
    "destination_longitude": "float",
    "destination_latitude": "float",
    "origin_h3_index": "string",
    "destination_h3_index": "string",
    "origin_time_utc": "datetime",
    "destination_time_utc": "datetime",
    "trip_id": "string",
    "movement_seq": "int",
}

BASE_FIELD_DTYPES = {
    "origin_municipality": "string",
    "destination_municipality": "string",
    "timezone_offset_min": "int",
    "origin_time_local_hhmm": "string",
    "destination_time_local_hhmm": "string",
    "trip_weight": "float",
    "mode_sequence": "string",
    "mode": "categorical",
    "purpose": "categorical",
    "day_type": "categorical",
    "time_period": "categorical",
    "user_gender": "categorical",
    "user_age_group": "categorical",
    "income_quintile": "categorical",
}

CANONICAL_DOMAINS = {
    "mode": [
        "walk", "bicycle", "scooter", "motorcycle", "car",
        "taxi", "ride_hailing", "bus", "metro", "train", "other",
    ],
    "purpose": [
        "home", "work", "education", "shopping", "errand",
        "health", "leisure", "transfer", "other",
    ],
    "day_type": ["weekday", "weekend", "holiday"],
    "time_period": ["night", "morning", "midday", "afternoon", "evening"],
    "user_gender": ["female", "male", "other", "unknown"],
    "user_age_group": ["0-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65-plus", "unknown"],
    "income_quintile": ["1", "2", "3", "4", "5", "unknown"],
}

RICH_BASE_FIELDS = [
    "origin_municipality",
    "destination_municipality",
    "timezone_offset_min",
    "origin_time_local_hhmm",
    "destination_time_local_hhmm",
    "trip_weight",
    "mode_sequence",
    "mode",
    "purpose",
    "day_type",
    "time_period",
    "user_gender",
    "user_age_group",
    "income_quintile",
]

RICH_EXTRA_COLUMNS = [
    "activity_status",
    "education_level",
    "travel_time_bucket",
    "season",
    "fare_payment_type",
    "bike_lane_usage",
    "home_tenure",
]

### 1.2 Builder de schema rica

Qué prepara: un TripSchema suficientemente rico para import + build + export.

In [52]:
def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=None,
        domain=domain,
    )


def make_rich_trip_schema() -> TripSchema:
    fields = {}

    for field_name in REQUIRED_FIELDS_ORDER:
        fields[field_name] = make_field(
            field_name,
            REQUIRED_FIELD_DTYPES[field_name],
            required=True,
        )

    for field_name, dtype_name in BASE_FIELD_DTYPES.items():
        domain = None
        if dtype_name == "categorical":
            domain = DomainSpec(values=CANONICAL_DOMAINS[field_name], extendable=True)

        fields[field_name] = make_field(
            field_name,
            dtype_name,
            required=False,
            domain=domain,
        )

    return TripSchema(
        version="1.1",
        fields=fields,
        required=list(REQUIRED_FIELDS_ORDER),
        semantic_rules=None,
    )

### 1.3 Builder de source dataframe rica

Qué prepara: dataframes de entrada con más filas y más columnas que los smoke tests.

In [53]:
def build_rich_source_dataframe(seed: int, filas: int) -> pd.DataFrame:
    df = generate_synthetic_trip_dataframe(
        filas=filas,
        seed=seed,
        duplicate_mode="none",
        tier_temporal="tier_1",
        tier1_datetime_format="utc_string_z",
        coord_format="numeric",
        h3_mode="provided_valid",
        trip_structure="multistage",
        max_movements_per_trip=3,
        base_fields=RICH_BASE_FIELDS,
        extra_value_domains={
            "mode": ["canon"],
            "purpose": ["canon"],
            "day_type": ["canon"],
            "time_period": ["canon"],
            "user_gender": ["canon"],
            "user_age_group": ["canon"],
            "income_quintile": ["canon"],
        },
        extra_columns=RICH_EXTRA_COLUMNS,
        null_ratio={
            "origin_municipality": 0.03,
            "destination_municipality": 0.03,
        },
    )
    return df

### 1.4 Helpers para materializar TripDataset ricas

Qué prepara: importación + validación mínima suficiente para usar las fixtures como input de build_flows.

In [54]:
def build_tripdataset_fixture(
    df: pd.DataFrame,
    schema: TripSchema,
    *,
    source_name: str,
    validate_after_import: bool = True,
) -> tuple[TripDataset, object, object | None]:
    trips, import_report = import_trips_from_dataframe(
        df,
        schema,
        source_name=source_name,
        options=ImportOptions(
            keep_extra_fields=True,
            selected_fields=None,
            strict=False,
            strict_domains=False,
            single_stage=False,
            source_timezone=None,
        ),
        provenance={
            "source": {"name": source_name, "entity": "trips"},
            "ingestion": {"created_at_utc": "2026-04-06T00:00:00Z"},
            "notes": [f"fixture de integración {source_name}"],
        },
        h3_resolution=12,
    )

    assert import_report.ok is True

    validation_report = None
    if validate_after_import:
        validation_report = validate_trips(
            trips,
            options=ValidationOptions(
                strict=False,
                validate_domains="off",
                validate_temporal_consistency=False,
                validate_duplicates=False,
            ),
        )
        assert validation_report.ok is True
        assert trips.metadata["is_validated"] is True

    return trips, import_report, validation_report

### 1.5 Fixtures base de integración

Qué prepara:

- tripdataset_validated_small
- tripdataset_ready_for_flows
- tripdataset_non_buildable
- builders para flowdataset_small y flowdataset_segmented

In [65]:
trip_schema_flows_rich = make_rich_trip_schema()

source_df_small = build_rich_source_dataframe(seed=20260406, filas=60)
source_df_rich = build_rich_source_dataframe(seed=20260407, filas=260)

tripdataset_validated_small, small_import_report, small_validation_report = build_tripdataset_fixture(
    source_df_small,
    trip_schema_flows_rich,
    source_name="synthetic_small_for_flows",
    validate_after_import=True,
)

tripdataset_ready_for_flows, rich_import_report, rich_validation_report = build_tripdataset_fixture(
    source_df_rich,
    trip_schema_flows_rich,
    source_name="synthetic_rich_for_flows",
    validate_after_import=True,
)

tripdataset_non_buildable = clone_tripdataset(tripdataset_validated_small)
tripdataset_non_buildable.data["origin_h3_index"] = pd.NA
tripdataset_non_buildable.data["destination_h3_index"] = pd.NA
tripdataset_non_buildable.metadata["is_validated"] = True


def make_flowdataset_small() -> tuple[FlowDataset, FlowBuildReport]:
    trips = clone_tripdataset(tripdataset_validated_small)
    return build_flows(
        trips,
        options=FlowBuildOptions(
            h3_resolution=8,
            group_by=None,
            time_aggregation="none",
            min_trips_per_flow=1,
            keep_flow_to_trips=False,
            require_validated=True,
        ),
    )


def make_flowdataset_segmented(*, h3_res=7, g_by=["mode", "purpose", "user_gender"], t_agg="day" , t_basis="origin") -> tuple[FlowDataset, FlowBuildReport]:
    trips = clone_tripdataset(tripdataset_ready_for_flows)
    return build_flows(
        trips,
        options=FlowBuildOptions(
            h3_resolution=h3_res,
            group_by=g_by,
            time_aggregation=t_agg,
            time_basis=t_basis,
            min_trips_per_flow=2,
            keep_flow_to_trips=False,
            require_validated=True,
        ),
    )

print("source_df_small.shape =", source_df_small.shape)
print("source_df_rich.shape  =", source_df_rich.shape)
print("tripdataset_validated_small.shape =", tripdataset_validated_small.data.shape)
print("tripdataset_ready_for_flows.shape =", tripdataset_ready_for_flows.data.shape)

show_ok("Sección 1 lista")

source_df_small.shape = (60, 33)
source_df_rich.shape  = (260, 33)
tripdataset_validated_small.shape = (60, 33)
tripdataset_ready_for_flows.shape = (260, 33)
OK - Sección 1 lista


## Sección 2. Integration tests de build_flows() / export_flows()

### Test 1 - build_flows feliz con fixture rica y contrato canónico interno

Qué prueba: caso principal correcto de build sobre un TripDataset rico y validado.
Verifica:

- construcción de FlowDataset,
- esquema interno canónico,
- aggregation_spec,
- provenance,
- metadata/evento,
- summary/parameters,
- y no mutación del TripDataset de entrada.

In [56]:
case_dir = make_case_dir("test_01_build_happy_rich")

trips = clone_tripdataset(tripdataset_ready_for_flows)
data_before = trips.data.copy(deep=True)
metadata_before = copy.deepcopy(trips.metadata)

flow_ds, report = build_flows(
    trips,
    options=FlowBuildOptions(
        h3_resolution=6,
        group_by=[ "purpose"],
        #time_aggregation="day",
        #time_basis="origin",
        min_trips_per_flow=2,
        keep_flow_to_trips=False,
        require_validated=True,
    ),
)

assert isinstance(flow_ds, FlowDataset)
assert isinstance(report, FlowBuildReport)
assert report.ok is True

display(trips.data)
display(report.issues)
display(flow_ds.flows)

assert len(flow_ds.flows) > 0
assert {"flow_id", "origin_h3_index", "destination_h3_index", "flow_count", "flow_value"}.issubset(flow_ds.flows.columns)
assert {"purpose"}.issubset(flow_ds.flows.columns)

assert flow_ds.aggregation_spec["h3_resolution"] == 6
assert flow_ds.aggregation_spec["group_by"] == ["purpose"]
#assert flow_ds.aggregation_spec["time_aggregation"] == "day"
assert "effective_flow_keys" in flow_ds.aggregation_spec

assert flow_ds.metadata["is_validated"] is False
assert flow_ds.metadata["events"][-1]["op"] == "build_flows"
assert flow_ds.metadata["events"][-1]["summary"] == report.summary
assert flow_ds.metadata["events"][-1]["parameters"] == report.parameters

assert "derived_from" in flow_ds.provenance
assert isinstance(flow_ds.provenance["derived_from"], list)
assert flow_ds.source_trips is trips

assert report.summary["n_trips_in"] == len(trips.data)
assert report.summary["n_flows_out"] == len(flow_ds.flows)
assert report.summary["n_flow_to_trips_rows"] is None

pd.testing.assert_frame_equal(trips.data, data_before)
assert trips.metadata["events"] == metadata_before["events"]

# display(flow_ds.flows.head(10))
display(report.summary)
show_ok("Test 1 - build feliz con fixture rica")

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,origin_municipality,destination_municipality,timezone_offset_min,origin_time_local_hhmm,destination_time_local_hhmm,trip_weight,mode_sequence,mode,purpose,day_type,time_period,user_gender,user_age_group,income_quintile,activity_status,education_level,travel_time_bucket,season,fare_payment_type,bike_lane_usage,home_tenure
0,m00000,u0061,tm_00000,0,-70.522181,-33.483310,-70.504064,-33.487758,2026-03-03 03:49:00+00:00,2026-03-03 05:01:00+00:00,8cb2c508420b3ff,8cb2c508504c1ff,Estación Central,Vitacura,-180,03:49,05:01,4.494,walk+car+car,taxi,other,weekend,midday,female,55-64,2,unemployed,secondary,41-60,vacation,free_transfer,sometimes,owned
1,m00001,u0057,tm_00001,0,-70.621575,-33.426532,-70.570535,-33.416125,2026-03-02 06:22:00+00:00,2026-03-02 07:54:00+00:00,8cb2c556aa2ddff,8cb2c519a195dff,Vitacura,La Florida,-180,06:22,07:54,4.009,bus+bus+car,car,shopping,holiday,midday,male,15-24,2,unemployed,postgraduate,0-10,summer,not_applicable,always,loaned
2,m00002,u0057,tm_00001,1,-70.714779,-33.490598,-70.737610,-33.486348,2026-03-01 20:03:00+00:00,2026-03-01 20:22:00+00:00,8cb2c555aaec3ff,8cb2c542d2621ff,<NA>,Santiago,-180,20:03,20:22,3.645,car+car+car,motorcycle,errand,holiday,evening,female,55-64,3,unemployed,secondary,41-60,normal_period,card,always,loaned
3,m00003,u0049,tm_00002,0,-70.785966,-33.403344,-70.790588,-33.409184,2026-03-02 20:41:00+00:00,2026-03-02 21:34:00+00:00,8cb2c550809cbff,8cb2c55086b29ff,San Miguel,Santiago,-180,20:41,21:34,1.914,walk,walk,other,weekend,morning,female,35-44,4,homemaker,postgraduate,21-40,normal_period,cash,not_applicable,rented
4,m00004,u0049,tm_00002,1,-70.752340,-33.544500,-70.712639,-33.568845,2026-03-05 19:44:00+00:00,2026-03-05 21:13:00+00:00,8cb2c54727663ff,8cb2c54466e99ff,Estación Central,Las Condes,-180,19:44,21:13,3.220,train,scooter,transfer,weekend,midday,female,45-54,4,unemployed,none,0-10,normal_period,cash,not_applicable,owned
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255,m00255,u0047,tm_00128,1,-70.726577,-33.346355,-70.746171,-33.358891,2026-03-02 13:32:00+00:00,2026-03-02 13:59:00+00:00,8cb2c552d22a1ff,8cb2c55282483ff,Ñuñoa,Ñuñoa,-180,13:32,13:59,3.331,bus,other,home,weekend,morning,other,unknown,5,studying,primary,21-40,normal_period,not_applicable,sometimes,loaned
256,m00256,u0047,tm_00128,2,-70.757270,-33.393845,-70.748706,-33.414003,2026-03-03 18:35:00+00:00,2026-03-03 20:06:00+00:00,8cb2c550d05b7ff,8cb2c5554d8c5ff,San Miguel,Providencia,-180,18:35,20:06,2.561,walk+car+bus,train,home,weekend,night,female,unknown,4,working,postgraduate,11-20,vacation,cash,never,owned
257,m00257,u0043,tm_00129,0,-70.611955,-33.431106,-70.620529,-33.482948,2026-03-03 16:45:00+00:00,2026-03-03 17:05:00+00:00,8cb2c556b99b5ff,8cb2c5549713bff,Recoleta,Estación Central,-180,16:45,17:05,1.657,walk+bus+bus,taxi,transfer,holiday,night,male,65-plus,5,unemployed,none,60+,winter,free_transfer,sometimes,rented
258,m00258,u0043,tm_00129,1,-70.552031,-33.510194,-70.524002,-33.490320,2026-03-08 00:38:00+00:00,2026-03-08 01:55:00+00:00,8cb2c50823b2dff,8cb2c5080846bff,Independencia,Maipú,-180,00:38,01:55,4.635,train+metro+walk,motorcycle,home,holiday,evening,other,55-64,5,studying,secondary,41-60,normal_period,card,never,owned


[]

,flow_id,origin_h3_index,destination_h3_index,purpose,flow_count,flow_value
0,flow_0000000,86b2c508fffffff,86b2c508fffffff,other,2,7.200
1,flow_0000001,86b2c5097ffffff,86b2c50b7ffffff,work,3,5.065
2,flow_0000002,86b2c518fffffff,86b2c519fffffff,shopping,2,4.158
3,flow_0000003,86b2c518fffffff,86b2c51afffffff,health,2,7.843
4,flow_0000004,86b2c5197ffffff,86b2c5197ffffff,transfer,2,8.467
5,flow_0000005,86b2c542fffffff,86b2c555fffffff,errand,2,7.863
6,flow_0000006,86b2c544fffffff,86b2c544fffffff,errand,2,5.892
7,flow_0000007,86b2c552fffffff,86b2c5507ffffff,home,2,6.720
8,flow_0000008,86b2c5547ffffff,86b2c5547ffffff,shopping,2,7.060
9,flow_0000009,86b2c5547ffffff,86b2c556fffffff,leisure,2,5.211


{'n_trips_in': 260,
 'n_trips_eligible': 260,
 'n_trips_dropped': 0,
 'n_flows_out': 16,
 'n_flow_to_trips_rows': None}

OK - Test 1 - build feliz con fixture rica


### Test 2 - export_flows feliz desde build segmentado y con extras explícitas

Qué prueba: export exitoso desde un FlowDataset construido por la API pública, preservando campos extra útiles para Flowmap City.

In [67]:
case_dir = make_case_dir("test_02_export_happy_from_build")

flow_ds, build_report = make_flowdataset_segmented(h3_res=5, g_by=["user_gender"])

export_result, export_report = export_flows(
    flow_ds,
    output_root=str(case_dir),
    options=ExportFlowsOptions(
        format="flowmap_blue",
        mode="error_if_exists",
        folder_name="segmented_export_case",
        extra_flow_fields=["user_gender", "window_start_utc"],
    ),
)

assert build_report.ok is True
assert isinstance(export_result, FlowExportResult)
assert isinstance(export_report, OperationReport)
assert export_report.ok is True

export_dir = Path(export_result.export_dir)
flows_csv_path = Path(export_result.artifacts["flows"])
locations_csv_path = Path(export_result.artifacts["locations"])
metadata_json_path = Path(export_result.artifacts["metadata"])

assert export_dir.exists()
assert flows_csv_path.exists()
assert locations_csv_path.exists()
assert metadata_json_path.exists()

flows_csv = pd.read_csv(flows_csv_path)
locations_csv = pd.read_csv(locations_csv_path)
sidecar = json.loads(metadata_json_path.read_text(encoding="utf-8"))

assert {"origin", "dest", "count", "user_gender", "window_start_utc"}.issubset(flows_csv.columns)
assert {"id", "name", "lat", "lon"}.issubset(locations_csv.columns)

assert export_report.summary["n_flows"] == len(flows_csv)
assert export_report.summary["n_locations"] == len(locations_csv)
assert export_report.summary["files_written"] == ["flows.csv", "locations.csv", "metadata.json"]

assert sidecar["artifact_type"] == "flow_export"
assert sidecar["format"] == "flowmap_blue"
assert sidecar["flow_dataset_ref"]["aggregation_spec"]["group_by"] == ["user_gender"]
assert sidecar["export"]["count_source"] == "flow_value"
assert sidecar["export"]["parameters"]["extra_flow_fields"] == ["user_gender", "window_start_utc"]

assert flow_ds.metadata["events"][-1]["op"] == "export_flows"
assert flow_ds.metadata["events"][-1]["summary"] == export_report.summary
assert flow_ds.metadata["events"][-1]["parameters"] == export_report.parameters

display(flows_csv.head(10))
display(locations_csv.head(10))
display(sidecar["export"])
show_ok("Test 2 - export feliz desde build segmentado")

,origin,dest,count,user_gender,window_start_utc
0,85b2c50bfffffff,85b2c50bfffffff,1.787,male,2026-03-02
1,85b2c50bfffffff,85b2c50bfffffff,3.480,male,2026-03-03
2,85b2c50bfffffff,85b2c50bfffffff,1.094,other,2026-03-03
3,85b2c50bfffffff,85b2c50bfffffff,1.384,female,2026-03-04
4,85b2c50bfffffff,85b2c50bfffffff,7.187,male,2026-03-04
5,85b2c50bfffffff,85b2c50bfffffff,13.496,unknown,2026-03-07
6,85b2c50bfffffff,85b2c51bfffffff,5.022,other,2026-03-02
7,85b2c51bfffffff,85b2c51bfffffff,10.049,male,2026-03-02
8,85b2c51bfffffff,85b2c51bfffffff,9.736,female,2026-03-03
9,85b2c51bfffffff,85b2c51bfffffff,7.776,unknown,2026-03-04


,id,name,lat,lon
0,85b2c50bfffffff,85b2c50bfffffff,-33.496369,-70.530593
1,85b2c51bfffffff,85b2c51bfffffff,-33.356983,-70.517466
2,85b2c543fffffff,85b2c543fffffff,-33.501549,-70.835463
3,85b2c547fffffff,85b2c547fffffff,-33.568667,-70.689561
4,85b2c553fffffff,85b2c553fffffff,-33.362051,-70.822135
5,85b2c557fffffff,85b2c557fffffff,-33.429365,-70.676326
6,85b2c5cffffffff,85b2c5cffffffff,-33.289783,-70.663106


{'parameters': {'output_root': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\build_flows\\tmp_integration\\test_02_export_happy_from_build',
  'export_dir': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\build_flows\\tmp_integration\\test_02_export_happy_from_build\\segmented_export_case',
  'format': 'flowmap_blue',
  'mode': 'error_if_exists',
  'folder_name': 'segmented_export_case',
  'extra_flow_fields': ['user_gender', 'window_start_utc']},
 'summary': {'n_flows': 48,
  'n_locations': 7,
  'files_written': ['flows.csv', 'locations.csv', 'metadata.json']},
 'count_source': 'flow_value'}

OK - Test 2 - export feliz desde build segmentado


### Test 3 - build_flows degradado: resultado vacío por threshold alto

Qué prueba: caso degradado donde sí había movements buildables, pero el umbral min_trips_per_flow deja 0 flujos y la operación retorna warning, no error.

In [68]:
case_dir = make_case_dir("test_03_build_empty_after_threshold")

trips = clone_tripdataset(tripdataset_validated_small)

flow_ds, report = build_flows(
    trips,
    options=FlowBuildOptions(
        h3_resolution=8,
        group_by=["mode", "purpose"],
        time_aggregation="none",
        min_trips_per_flow=len(trips.data) + 10,
        keep_flow_to_trips=False,
        require_validated=True,
    ),
)

codes = [issue.code for issue in report.issues]

assert report.ok is True
assert len(flow_ds.flows) == 0
assert "FLOW.OUTPUT.EMPTY_AFTER_THRESHOLD" in codes

assert report.summary["n_trips_in"] == len(trips.data)
assert report.summary["n_trips_eligible"] > 0
assert report.summary["n_flows_out"] == 0

assert flow_ds.metadata["events"][-1]["op"] == "build_flows"

display(report.issues)
display(report.summary)
show_ok("Test 3 - build degradado vacío por threshold alto")

[Issue(level='warning', code='FLOW.OUTPUT.EMPTY_AFTER_THRESHOLD', message='La agregación finalizó sin flujos luego de aplicar min_trips_per_flow=70.', field=None, source_field=None, row_count=None, details={'check': 'output', 'min_trips_per_flow': 70, 'n_trips_in': 60, 'n_trips_eligible': 60, 'n_flows_out': 0, 'reason': 'empty_after_threshold'})]

{'n_trips_in': 60,
 'n_trips_eligible': 60,
 'n_trips_dropped': 0,
 'n_flows_out': 0,
 'n_flow_to_trips_rows': None}

OK - Test 3 - build degradado vacío por threshold alto


### Test 4 - build_flows fatal: dataset no construible por ausencia total de H3 OD

Qué prueba: caso fatal donde ningún movement puede entrar a agregación porque todos quedaron sin ambos H3.

In [69]:
case_dir = make_case_dir("test_04_build_fatal_non_buildable")

trips = clone_tripdataset(tripdataset_non_buildable)

try:
    _ = build_flows(
        trips,
        options=FlowBuildOptions(
            h3_resolution=8,
            group_by=None,
            time_aggregation="none",
            min_trips_per_flow=1,
            keep_flow_to_trips=False,
            require_validated=True,
        ),
    )
    raise AssertionError("Se esperaba ValidationError por dataset no construible.")
except ValidationError as exc:
    assert exc.code == "FLOW.OUTPUT.NO_BUILDABLE_MOVEMENTS"

show_ok("Test 4 - build fatal por dataset no construible")

OK - Test 4 - build fatal por dataset no construible


### Test 5 - export_flows fatal: FlowDataset no exportable

Qué prueba: caso fatal donde el FlowDataset.flows no cumple el mínimo interno exportable.

In [70]:
case_dir = make_case_dir("test_05_export_fatal_non_exportable")

flow_ds, build_report = make_flowdataset_small()
assert build_report.ok is True

bad_flow_ds = clone_flowdataset(flow_ds)
bad_flow_ds.flows = bad_flow_ds.flows.drop(columns=["flow_value"])

try:
    _ = export_flows(
        bad_flow_ds,
        output_root=str(case_dir),
        options=ExportFlowsOptions(
            format="flowmap_blue",
            mode="error_if_exists",
            folder_name="bad_export_case",
            extra_flow_fields=None,
        ),
    )
    raise AssertionError("Se esperaba ExportError por FlowDataset no exportable.")
except ExportError as exc:
    assert exc.code == "EXPORT_FLOWS.DATA.REQUIRED_FIELDS_MISSING"

show_ok("Test 5 - export fatal con flow no exportable")

OK - Test 5 - export fatal con flow no exportable


### Test 6 - consistencia de metadata, eventos y summaries del bloque build -> export

Qué prueba: que el bloque Trip → Flow deje evidencia consistente y append-only en metadata y sidecar.

In [71]:
case_dir = make_case_dir("test_06_metadata_events_summaries")

trips = clone_tripdataset(tripdataset_ready_for_flows)
initial_trip_events = copy.deepcopy(trips.metadata.get("events", []))

flow_ds, build_report = build_flows(
    trips,
    options=FlowBuildOptions(
        h3_resolution=8,
        group_by=["mode"],
        time_aggregation="week",
        time_basis="origin",
        min_trips_per_flow=2,
        keep_flow_to_trips=False,
        require_validated=True,
    ),
)

assert build_report.ok is True
assert flow_ds.metadata["events"][-1]["op"] == "build_flows"
assert flow_ds.metadata["events"][-1]["summary"] == build_report.summary
assert flow_ds.metadata["events"][-1]["parameters"] == build_report.parameters
assert "issues_summary" in flow_ds.metadata["events"][-1]

export_result, export_report = export_flows(
    flow_ds,
    output_root=str(case_dir),
    options=ExportFlowsOptions(
        format="flowmap_blue",
        mode="error_if_exists",
        folder_name="metadata_chain_case",
        extra_flow_fields=["mode", "window_start_utc"],
    ),
)

assert export_report.ok is True
assert flow_ds.metadata["events"][-2]["op"] == "build_flows"
assert flow_ds.metadata["events"][-1]["op"] == "export_flows"
assert flow_ds.metadata["events"][-1]["summary"] == export_report.summary
assert flow_ds.metadata["events"][-1]["parameters"] == export_report.parameters
assert "issues_summary" in flow_ds.metadata["events"][-1]

sidecar = json.loads(Path(export_result.artifacts["metadata"]).read_text(encoding="utf-8"))
assert sidecar["flow_dataset_ref"]["dataset_id"] == flow_ds.metadata["dataset_id"]
assert sidecar["flow_dataset_ref"]["metadata"]["events"][-1]["op"] == "build_flows"
assert_json_safe(sidecar, "flow export sidecar")

# build_flows no debe mutar el metadata del TripDataset fuente
assert trips.metadata.get("events", []) == initial_trip_events

display(flow_ds.metadata["events"][-2:])
display(build_report.summary)
display(export_report.summary)
show_ok("Test 6 - metadata/eventos/summaries del bloque")

[{'op': 'build_flows',
  'ts_utc': '2026-04-06T05:59:48.444643Z',
  'parameters': {'h3_resolution': 8,
   'group_by': ['mode'],
   'time_aggregation': 'week',
   'time_basis': 'origin',
   'min_trips_per_flow': 2,
   'keep_flow_to_trips': False,
   'require_validated': True,
   'strict': False,
   'max_issues': 1000},
  'summary': {'n_trips_in': 260,
   'n_trips_eligible': 260,
   'n_trips_dropped': 0,
   'n_flows_out': 0,
   'n_flow_to_trips_rows': None},
  'issues_summary': {'counts': {'info': 0, 'warning': 1, 'error': 0},
   'top_codes': [{'code': 'FLOW.OUTPUT.EMPTY_AFTER_THRESHOLD', 'count': 1}]}},
 {'op': 'export_flows',
  'ts_utc': '2026-04-06T05:59:48.467833Z',
  'parameters': {'output_root': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\build_flows\\tmp_integration\\test_06_metadata_events_summaries',
   'export_dir': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\build_flows\\tmp_integration\\test_06_metadata_events_summaries\\metadata_chain_case',
   'format': 'f

{'n_trips_in': 260,
 'n_trips_eligible': 260,
 'n_trips_dropped': 0,
 'n_flows_out': 0,
 'n_flow_to_trips_rows': None}

{'n_flows': 0,
 'n_locations': 0,
 'files_written': ['flows.csv', 'locations.csv', 'metadata.json']}

OK - Test 6 - metadata/eventos/summaries del bloque


### Test 7 - build_flows con flow_to_trips y export sin persistir backlinks como artefacto

Qué prueba: drill-down mínimo por flow_to_trips y confirmación de que OP-09 no lo exporta como tabla separada en v1.1.

In [72]:
case_dir = make_case_dir("test_07_flow_to_trips")

trips = clone_tripdataset(tripdataset_validated_small)

flow_ds, build_report = build_flows(
    trips,
    options=FlowBuildOptions(
        h3_resolution=8,
        group_by=["mode"],
        time_aggregation="none",
        min_trips_per_flow=1,
        keep_flow_to_trips=True,
        require_validated=True,
    ),
)

assert build_report.ok is True
assert flow_ds.flow_to_trips is not None
assert set(flow_ds.flow_to_trips.columns) == {"flow_id", "movement_id"}
assert build_report.summary["n_flow_to_trips_rows"] == len(flow_ds.flow_to_trips)

# Con min_trips_per_flow=1, cada trip buildable debe aparecer en flow_to_trips
assert len(flow_ds.flow_to_trips) == build_report.summary["n_trips_eligible"]

export_result, export_report = export_flows(
    flow_ds,
    output_root=str(case_dir),
    options=ExportFlowsOptions(
        format="flowmap_blue",
        mode="error_if_exists",
        folder_name="flow_to_trips_case",
        extra_flow_fields=["mode"],
    ),
)

assert export_report.ok is True
assert set(export_result.artifacts.keys()) == {"flows", "locations", "metadata"}
assert not (Path(export_result.export_dir) / "flow_to_trips.csv").exists()

display(flow_ds.flow_to_trips.head(10))
display(pd.read_csv(export_result.artifacts["flows"]).head(10))
show_ok("Test 7 - flow_to_trips presente en build y ausente como artefacto de export")

,flow_id,movement_id
0,flow_0000042,m00000
1,flow_0000047,m00001
2,flow_0000040,m00002
3,flow_0000030,m00003
4,flow_0000048,m00004
5,flow_0000022,m00005
6,flow_0000041,m00006
7,flow_0000012,m00007
8,flow_0000039,m00008
9,flow_0000044,m00009


,origin,dest,count,mode
0,88b2c50149fffff,88b2c51159fffff,2.238,metro
1,88b2c50183fffff,88b2c50193fffff,3.421,car
2,88b2c50807fffff,88b2c5095dfffff,3.889,bus
3,88b2c50813fffff,88b2c50819fffff,1.271,bus
4,88b2c50849fffff,88b2c50a1dfffff,2.099,ride_hailing
5,88b2c50857fffff,88b2c50913fffff,0.891,car
6,88b2c50917fffff,88b2c50925fffff,2.803,train
7,88b2c5092bfffff,88b2c509c9fffff,2.672,taxi
8,88b2c50949fffff,88b2c54609fffff,2.916,car
9,88b2c509ebfffff,88b2c509a9fffff,3.400,motorcycle


OK - Test 7 - flow_to_trips presente en build y ausente como artefacto de export


### Inspección visible de artefactos creados

Qué prueba: deja visible qué directorios/casos quedaron escritos.

In [73]:
# ### Test 8 - inspección visible de artefactos creados
# Qué prueba: inspección manual rápida del árbol de artefactos creados durante integración.

for path in sorted(IT_ROOT.rglob("*")):
    rel = path.relative_to(IT_ROOT)
    kind = "DIR " if path.is_dir() else "FILE"
    print(f"{kind:4} {rel}")

DIR  test_01_build_happy_rich
DIR  test_02_export_happy_from_build
DIR  test_02_export_happy_from_build\segmented_export_case
FILE test_02_export_happy_from_build\segmented_export_case\flows.csv
FILE test_02_export_happy_from_build\segmented_export_case\locations.csv
FILE test_02_export_happy_from_build\segmented_export_case\metadata.json
DIR  test_03_build_empty_after_threshold
DIR  test_04_build_fatal_non_buildable
DIR  test_05_export_fatal_non_exportable
DIR  test_06_metadata_events_summaries
DIR  test_06_metadata_events_summaries\metadata_chain_case
FILE test_06_metadata_events_summaries\metadata_chain_case\flows.csv
FILE test_06_metadata_events_summaries\metadata_chain_case\locations.csv
FILE test_06_metadata_events_summaries\metadata_chain_case\metadata.json
DIR  test_07_flow_to_trips
DIR  test_07_flow_to_trips\flow_to_trips_case
FILE test_07_flow_to_trips\flow_to_trips_case\flows.csv
FILE test_07_flow_to_trips\flow_to_trips_case\locations.csv
FILE test_07_flow_to_trips\flow_to_t